In [1]:
!apt-get -y install protobuf-compiler

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
protobuf-compiler is already the newest version (3.12.4-1ubuntu7.22.04.4).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [3]:
!pip install  flwr "flwr[simulation]" "protobuf" "cryptography<44" -q

In [4]:
import copy, torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import numpy as np
from collections import defaultdict
import timm

In [5]:
class ViTEncoder(nn.Module):

  """
  Lightweight ViT encoder using timm.
  deit_tiny_patch16_224 gives strong features at low compute cost.
  For heavier compute: vit_small_patch16_224 or vit_base_patch16_224
  """

  def __init__(self, model_name='deit_tiny_patch16_224',
  pretrained=True, out_dim=192):
    super().__init__()
    self.backbone = timm.create_model(
    model_name, pretrained=pretrained,
    num_classes=0, # remove classification head
    global_pool='avg' # average pool CLS token
    )
    self.out_dim = self.backbone.num_features

  def forward(self, x):
    # timm expects 224x224; add resize in transforms
    return self.backbone(x) # (B, out_dim)

class ProjectionHead(nn.Module):
    """2-layer MLP projection head"""
    def __init__(self, in_dim, hidden_dim=128, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x):
      return self.net(x)

class MOONModel(nn.Module):
  def __init__(self, encoder, proj_dim=256, num_classes=10):

    super().__init__()
    self.encoder = encoder
    enc_dim = encoder.out_dim # 192 for deit_tiny
    self.proj_head = ProjectionHead(enc_dim, 256, proj_dim)
    self.classifier = nn.Linear(enc_dim, num_classes)

  def forward(self, x):
    rep = self.encoder(x)
    return self.classifier(rep), self.proj_head(rep)

In [6]:
from torch.nn.functional import cosine_similarity as sim
def MOON_constrastive_loss(z, z_glob, z_prev, temperature=0.5):
  """
    l_con = -log[ exp(sim(z, z_glob)/τ) /
                  (exp(sim(z, z_glob)/τ) + exp(sim(z, z_prev)/τ)) ]

    pos_sim represents the similarity between the global projection and local (current) projection.
    neg_sim represents the similarity between the local (current) projection and local (previous) projection.

    We have to increase the similarity between the global projection and local (current) projection,
    decrease the similarity between the local (current) projection and local (previous) projection.
  """

  pos_sim = sim(z, z_glob, dim=-1) / temperature
  neg_sim = sim(z, z_prev, dim=-1) / temperature

  logits = torch.cat([pos_sim.reshape(-1,1), neg_sim.reshape(-1,1)], dim=1)

  #The labels is used to selected to maximise the pos_sim by guiding it's in the zeroth position.
  labels = torch.zeros(z.size(0)).long().to(z.device)

  #Since the cross entropy loss consist of both the softmax and logloss it is directly employed here to reduce the boilerplate code.
  con_loss = F.cross_entropy(logits, labels)

  return con_loss

In [7]:
def dirichlet_partition(dataset, num_parties, beta=0.5, seed=42):
    np.random.seed(seed)
    labels = np.array([dataset[i][1] for i in range(len(dataset))])
    class_indices = defaultdict(list)
    for idx, lbl in enumerate(labels): class_indices[lbl].append(idx)

    party_indices = defaultdict(list)
    for cls, idx_cls in class_indices.items():
        idx_cls = np.array(idx_cls); np.random.shuffle(idx_cls)
        props = np.random.dirichlet([beta] * num_parties)
        props = (props * len(idx_cls)).astype(int)
        props[-1] = len(idx_cls) - props[:-1].sum()
        start = 0
        for pid, count in enumerate(props):
            party_indices[pid].extend(idx_cls[start:start+count].tolist())
            start += count
    return dict(party_indices)

In [8]:
import flwr as fl
from torch.utils.data import DataLoader
import torch.nn.functional as F
import copy

class MOONClient(fl.client.NumPyClient):
  def __init__(self, model, train_loader, device, mu, temperature):
    self.model = model
    self.train_loader = train_loader
    self.device = device
    self.mu = mu
    self.temperature = temperature
    self.previous_model = None

  def get_parameters(self, config):
    """Return the current local model parameters"""
    return [val.cpu().numpy() for val in self.model.state_dict().values()]

  def set_parameters(self,parameters):
    """Set the current local model parameters"""

    keys = list(self.model.state_dict().keys())
    state = dict(zip(keys, [torch.tensor(val) for val in parameters]))
    self.model.load_state_dict(state, strict=True)
    self.model.to(self.device)

  def fit(self, parameters, config):

    # 1. Setup global and previous models
    global_model = copy.deepcopy(self.model)
    self.set_parameters(parameters)
    global_model.load_state_dict(self.model.state_dict())
    global_model.eval() # Good practice to freeze the global model

    if self.previous_model == None:
      self.previous_model = copy.deepcopy(self.model)
    self.previous_model.eval() # Good practice to freeze the previous model

    optimizer = torch.optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-5)
    self.model.train()

    total = correct = 0
    running_loss = 0.0

    # 2. Single training loop
    for _ in range(config.get("local_epochs",10)):
      for x,y in self.train_loader:
        x,y = x.to(self.device), y.to(self.device)

        # Forward pass
        logits,z = self.model(x)
        loss_sup = F.cross_entropy(logits, y)

        with torch.no_grad():
          _, z_glob = global_model(x)
          _, Z_prev = self.previous_model(x)

        # Calculate MOON loss
        loss_con = MOON_constrastive_loss(z, z_glob, Z_prev, self.temperature)
        loss = loss_sup + self.mu * loss_con

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track metrics
        with torch.no_grad():
          correct += (logits.argmax(1) == y).sum().item()
          total += y.size(0)
          running_loss += loss.item()

    # 3. Update previous model for the next round
    self.previous_model = copy.deepcopy(self.model)

    train_acc = correct / total if total > 0 else 0.0

    # Return results
    return self.get_parameters(config={}), len(self.train_loader.dataset), {
      "train_accuracy": train_acc,
      "train_loss": running_loss / total,
    }


  def evaluate(self, parameters, config):

    self.set_parameters(parameters)
    self.model.eval()
    loss = total = correct = 0

    with torch.no_grad(): # Added parentheses to torch.no_grad
      for x,y in self.train_loader:
        x,y = x.to(self.device), y.to(self.device)
        logits, _ = self.model(x)
        loss += F.cross_entropy(logits, y, reduction="sum").item() # Added reduction="sum" and .item() to loss
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)

    return float(loss), total, {"accuracy": correct/total}

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
from flwr.server.strategy import FedAvg

def weighted_average(metrics):
  total = sum(n for n, _ in metrics)
  acc = sum(n * m['accuracy'] for n, m in metrics) / total
  return {'accuracy': acc}

def weighted_train_average(metrics):
  total = sum(n for n, _ in metrics)
  acc = sum(n * m.get('train_accuracy', 0) for n, m in metrics) / total
  loss_avg = sum(n * m.get('train_loss', 0) for n, m in metrics) / total
  return {'train_accuracy': acc, 'train_loss': loss_avg}

strategy = FedAvg(
    fraction_fit                    = 1.0,
    fraction_evaluate               = 1.0,
    min_fit_clients                 = 3,
    min_evaluate_clients            = 3,
    min_available_clients           = 3,
    on_fit_config_fn                = lambda rnd: {"local_epochs": 2},
    fit_metrics_aggregation_fn = weighted_train_average,
    evaluate_metrics_aggregation_fn = weighted_average,
)

In [12]:
import flwr as fl
import torchvision, torchvision.transforms as T
from flwr.common import Context  # ← add this import

#Changed transformation w.r.t ViT
transforms = T.Compose([T.Resize(224),
 T.ToTensor(),
 T.Normalize((0.485,0.456,0.406), (0.229,0.224,0.225)) # ImageNet stats
])
train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms)

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_PARTIES = 3

party_data  = dirichlet_partition(train_set, NUM_PARTIES, beta=0.5)


def client_fn(context: Context):  # ← Context instead of cid
    cid = int(context.node_config["partition-id"])  # ← extract cid from context
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = MOONModel(ViTEncoder(), proj_dim=256, num_classes=10).to(device)
    loader = DataLoader(
        torch.utils.data.Subset(train_set, party_data[cid]),
        batch_size=64, shuffle=True, drop_last=True
    )
    return MOONClient(model, loader, device, mu=5, temperature=0.5).to_client()


fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_PARTIES,
    config=fl.server.ServerConfig(num_rounds=3),
    strategy=strategy,
    client_resources= {"num_cpus":1, "num_gpus":0.5}
)

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower simulation, config: num_rounds=3, no round_timeout
2026-03-06 05:14:55,847	INFO worker.py:1771 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      Flower VCE: Ray initialized with resources: {'node:__internal_head__': 1.0, 'CPU': 2.0, 'memory': 7866315572.0, 'nod

History (loss, distributed):
	round 1: 5597.844733498036
	round 2: 2370.5206516391954
	round 3: 1487.0220822291496
History (metrics, distributed, fit):
{'train_accuracy': [(1, 0.8363476061915488),
                    (2, 0.9280571239077585),
                    (3, 0.9518447643508086)],
 'train_loss': [(1, 0.06169024943169088),
                (2, 0.057433798150464),
                (3, 0.056339389998466986)]}
History (metrics, distributed, evaluate):
{'accuracy': [(1, 0.8945112179487179),
              (2, 0.9554086538461538),
              (3, 0.9692908653846154)]}

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device